In [0]:
# Databricks notebook source

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType, DateType, BooleanType, StringType
from pyspark.sql.window import Window

def transform_product(product_df):

    windowSpec_rw = Window.partitionBy("ProductID").orderBy("ModifiedDate")
    product_df = product_df.withColumn("Color",F.when(F.col("Color").isNull(), F.lit("N/A")).otherwise(F.col("Color"))).withColumn("Weight",F.when(F.col("Weight").isNull(), 0).otherwise(F.col("Weight"))).filter(F.col("StandardCost") > 0 ).filter(F.col("ListPrice") > 0 ).withColumn(
		"rw", F.row_number().over(windowSpec_rw))
    product_df = product_df.filter(F.col("rw") == 1).drop("rw")
    product_df = product_df.withColumn("processed_timestamp", F.current_timestamp())
    product_df = product_df.select(       
      F.col("ProductID").cast(IntegerType()).alias("ProductID"),
      F.initcap(F.trim(F.col("Name"))).cast(StringType()).alias("Name"),
      F.initcap(F.trim(F.col("ProductNumber"))).cast(StringType()).alias("ProductNumber"),
      F.col("MakeFlag").cast(BooleanType()).alias("MakeFlag"),
      F.col("FinishedGoodsFlag").cast(BooleanType()).alias("FinishedGoodsFlag"),
      F.initcap(F.trim(F.col("Color"))).cast(StringType()).alias("Color"),
      F.col("SafetyStockLevel").cast(IntegerType()).alias("SafetyStockLevel"),
      F.col("ReorderPoint").cast(IntegerType()).alias("ReorderPoint"),
      F.col("StandardCost").cast(DecimalType(19,4)).alias("StandardCost"),
      F.col("ListPrice").cast(DecimalType(19,4)).alias("ListPrice"),
      F.col("Size").cast(StringType()).alias("Size"),
      F.col("SizeUnitMeasureCode").cast(StringType()).alias("SizeUnitMeasureCode"),
      F.col("WeightUnitMeasureCode").cast(StringType()).alias("WeightUnitMeasureCode"),
      F.col("Weight").alias("Weight"),
      F.col("DaysToManufacture").cast(IntegerType()).alias("DaysToManufacture"),
      F.col("ProductLine").cast(StringType()).alias("ProductLine"),
      F.col("Class").cast(StringType()).alias("Class"),
      F.col("Style").cast(StringType()).alias("Style"),
      F.col("ProductSubcategoryID").cast(IntegerType()).alias("ProductSubcategoryID"),
      F.col("ProductModelID").cast(IntegerType()).alias("ProductModelID"),
      F.col("SellStartDate").cast(DateType()).alias("SellStartDate"),
      F.col("SellEndDate").cast(DateType()).alias("SellEndDate"),
      F.col("DiscontinuedDate").cast(DateType()).alias("DiscontinuedDate"),
      F.col("rowguid").cast(StringType()).alias("rowguid"),
      F.col("ModifiedDate").cast(DateType()).alias("ModifiedDate"),
      F.col("_rescued_data").cast(StringType()).alias("_rescued_data"),
      F.col("processed_timestamp")
    )
                                 
    return product_df




if __name__ == "__main__":

    product_tbl = dbutils.widgets.get("product")
    product_df = df = spark.read.table(product_tbl)
    product_df_tgt = transform_product(product_df)
    display(product_df_tgt)